### Setting up envs and imports


In [18]:
from dotenv import load_dotenv
import os
from pageindex import PageIndexClient
import pageindex.utils as utils

load_dotenv()
PAGE_INDEX_API_KEY = os.getenv("API_KEY")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
pi_client = PageIndexClient(api_key="PAGE_INDEX_API_KEY")
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
REGISTRY_PATH = "data/pdf_registry.json"
# print(pi_client)

if not PAGE_INDEX_API_KEY:
    print(
        "❌ Validation Failed: No 'API_KEY' entry detected within your local .env configuration."
    )
    exit(1)

clean_key = PAGE_INDEX_API_KEY.strip().replace('"', "").replace("'", "")
pi_client = PageIndexClient(api_key=clean_key)


### Testing Groq API KEY


In [4]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)
completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "What is the Capital of India?"}],
    reasoning_effort="medium",
)

print(completion.choices[0].message.content)


The capital of India is **New Delhi**.


### Testing NVIDIA NIM API Key


In [17]:
from openai import OpenAI
import os

# Load from environment variable
NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    print("❌ Error: NIM_API_KEY not found in environment variables")
    exit(1)

client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY)

completion = client.chat.completions.create(
    model="meta/llama-3.1-8b-instruct",
    messages=[{"role": "user", "content": "What is the Capital of India?"}],
    temperature=0.5,
    max_tokens=1024,
)

print(completion.choices[0].message.content)


The capital of India is New Delhi.


### Submitting the pdfs to the pi_client


In [6]:
import os

# finds every .pdf in your data/ folder automatically
pdf_files = [f for f in os.listdir("data") if f.endswith(".pdf")]

doc_ids = {}  # stores filename → doc_id

for pdf_file in pdf_files:
    pdf_path = os.path.join("data", pdf_file)
    response = pi_client.submit_document(pdf_path)
    doc_id = response["doc_id"]
    doc_ids[pdf_file] = doc_id
    print(f"✅ Submitted: {pdf_file} → {doc_id}")

print("\nAll doc_ids:", doc_ids)


✅ Submitted: LIC_Jeevan_Shagun_front_r.pdf → pi-cmrgj8ifd000i01qrtrq9002o
✅ Submitted: LIC_Jeevan_Shagun_Policy_inside_r.pdf → pi-cmrgj8jsv000j01qr7tefecxu
✅ Submitted: Policy-Document_LIC-s_New-Jeevan_Amar.pdf → pi-cmrgj8o7x000k01qr3dw0m60w
✅ Submitted: Accidental-Death-Benefit-101B001V01.pdf → pi-cmrgj8qzp000l01qrz5eezgxg
✅ Submitted: ACCIDENT_SHIELD_POLICY_Policy_Wording_2982ea20b0.pdf → pi-cmrgj8ulw000m01qroweet8jf
✅ Submitted: accident_super_guard_plus_policy_wordings_80ae144608.pdf → pi-cmrgj8xws000n01qry913n8s2
✅ Submitted: Arogya_Sanjeevani_Policy_Wording_KEN_034037d936.pdf → pi-cmrgj90gz000o01qrdqxly7m6

All doc_ids: {'LIC_Jeevan_Shagun_front_r.pdf': 'pi-cmrgj8ifd000i01qrtrq9002o', 'LIC_Jeevan_Shagun_Policy_inside_r.pdf': 'pi-cmrgj8jsv000j01qr7tefecxu', 'Policy-Document_LIC-s_New-Jeevan_Amar.pdf': 'pi-cmrgj8o7x000k01qr3dw0m60w', 'Accidental-Death-Benefit-101B001V01.pdf': 'pi-cmrgj8qzp000l01qrz5eezgxg', 'ACCIDENT_SHIELD_POLICY_Policy_Wording_2982ea20b0.pdf': 'pi-cmrgj8ulw000m01

In [25]:
import json
def load_registry() -> dict:
    if not os.path.exists(REGISTRY_PATH):
        return {}

    with open(REGISTRY_PATH, "r") as f:
        content = f.read().strip()

    if not content:  # file exists but is empty
        return {}

    return json.loads(content)

def save_to_registry(doc_id: str, pdf_path: str, description: str):
    registry = load_registry()
    registry[doc_id] = {
        "doc_id": doc_id,
        "filename": os.path.basename(pdf_path),
        "description": description,
    }
    with open(REGISTRY_PATH, "w") as f:
        json.dump(registry, f, indent=2)
    print(f"✅ Saved: {os.path.basename(pdf_path)} → {doc_id}")


In [20]:
def call_nim(prompt: str) -> str:
    client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY)
    completion = client.chat.completions.create(
        model="meta/llama-3.1-70b-instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return completion.choices[0].message.content # pyright: ignore[reportReturnType]


In [22]:
import time
import json


def generate_description(doc_id: str, filename: str) -> str:
    """
    Waits for PageIndex to finish processing, then feeds the node tree
    to NIM to auto-generate a registry description for that PDF.
    """
    print(f"⏳ Waiting for tree: {filename}...")
    while not pi_client.is_retrieval_ready(doc_id):
        time.sleep(3)

    tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
    tree_without_text = utils.remove_fields(tree.copy(), fields=["text"])

    prompt = f"""You are given a tree structure of an insurance PDF document.
Each node has a title and summary describing what that section covers.

Write a single description (2-3 sentences) that answers:
"What specific questions can a user ask that this document would answer?"

Be specific — mention coverage types, procedures, limits, rules covered.
Do NOT write generic phrases like "this document covers various insurance topics."

Document tree:
{json.dumps(tree_without_text, indent=2)}

Reply with ONLY the description text, nothing else.
"""
    description = call_nim(prompt)
    print(f"✅ {filename}:\n   {description}\n")
    return description


In [26]:
for filename, doc_id in doc_ids.items():
    description = generate_description(doc_id, filename)
    save_to_registry(
        doc_id=doc_id, pdf_path=os.path.join("data", filename), description=description
    )

print("\n🎉 Registry complete. Contents:")
print(json.dumps(load_registry(), indent=2))


⏳ Waiting for tree: LIC_Jeevan_Shagun_front_r.pdf...
✅ LIC_Jeevan_Shagun_front_r.pdf:
   This document would answer questions about the coverage and benefits of the 'Jeevan Shagun (With Profits)' plan, such as what conditions must be met for benefit payouts, what statutory terms and definitions apply, and what data is included in the policy schedule. It would also provide information on the claims process, including the legal framework and administrative contact details. Additionally, users can find answers to questions about policyholder and nominee data, as well as plan-related details.

✅ Saved: LIC_Jeevan_Shagun_front_r.pdf → pi-cmrgj8ifd000i01qrtrq9002o
⏳ Waiting for tree: LIC_Jeevan_Shagun_Policy_inside_r.pdf...
✅ LIC_Jeevan_Shagun_Policy_inside_r.pdf:
   This document answers questions about the roles of individuals involved in an insurance policy, such as the policyholder, nominee, and assignee, as well as technical policy parameters like commencement date, maturity, premiums, 

### User input the query


In [ ]:
user_query = input("What is your Query? \n")
print(user_query)